In [ ]:
# Phase 10 pilot: can a frozen DINOv3 ViT-L/16 turn our prepped studies into features fast enough, and does the
# checkpoint load for real? 200 studies, 9 windows x 6 slots at 336px. Pass conditions: NOTES 2026-10-06.
import glob, os, shutil, sys, time

GIT_SHA = '53b347a'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import numpy as np
import pandas as pd
import timm
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}, timm {timm.__version__}, torch {torch.__version__}')
assert (major, minor) >= (7, 0), 'need T4, not P100'

from knee.dataset import PreppedSlotDataset
from knee.dicom import SLOTS
from knee.features import PooledBackbone, extract_study_features, load_feature_cache, save_feature_cache
assert any('dinov3' in n for n in timm.list_models()), 'this timm has no DINOv3'
print('src carries the Phase 10 feature modules | GIT_SHA', GIT_SHA)


In [ ]:
# The prepped corpus, mounted from the four prep9 shard kernels (as every confirm kernel does).
PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)
print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts')
assert len(uid_to_npz) > 4000, 'shards do not cover the corpus'

slot_names = [n for n, _, _ in SLOTS]
assert len(slot_names) == 6
LIMIT = 200
rng = np.random.default_rng(0)
all_uids = sorted(uid_to_npz)
uids = sorted(rng.choice(all_uids, size=LIMIT, replace=False).tolist())
print(f'pilot sample: {len(uids)} studies (seed 0)')


In [ ]:
# The checkpoint. No internet here, so the weights come from the Kaggle Model; strict loading is the point:
# a silent partial load would leave random layers and every downstream number would be noise.
files = sorted(p for p in glob.glob('/kaggle/input/**/*', recursive=True)
               if os.path.isfile(p) and '/dinov3' in p.lower())
for p in files[:30]:
    print(f'  {p}  {os.path.getsize(p) / 2**20:.0f} MB')
weights = [p for p in files if p.endswith(('.safetensors', '.pth', '.pt', '.bin'))]
assert weights, 'no weight file found under the DINOv3 model mount'
if len(weights) > 1:
    prefer = [p for p in weights if any(k in os.path.basename(p).lower() for k in ('vitl', 'large'))]
    assert len(prefer) == 1, f'ambiguous weight files, pick one by hand: {weights}'
    weights = prefer
WEIGHTS = weights[0]
print('weights:', WEIGHTS)

if WEIGHTS.endswith('.safetensors'):
    from safetensors.torch import load_file
    state = load_file(WEIGHTS)
else:
    state = torch.load(WEIGHTS, map_location='cpu', weights_only=True)
for wrapper in ('state_dict', 'model', 'teacher'):
    if isinstance(state, dict) and wrapper in state and isinstance(state[wrapper], dict):
        state = state[wrapper]
for prefix in ('module.', 'model.', 'backbone.'):
    if state and all(k.startswith(prefix) for k in state):
        state = {k[len(prefix):]: v for k, v in state.items()}
print(f'{len(state)} tensors, first keys: {list(state)[:6]}')

from timm.models.eva import checkpoint_filter_fn
model = None
for name in ('vit_large_patch16_dinov3', 'vit_large_patch16_dinov3_qkvb'):
    for convert in (False, True):
        candidate = timm.create_model(name, pretrained=False, num_classes=0, img_size=336)
        sd = checkpoint_filter_fn(dict(state), candidate) if convert else dict(state)
        try:
            candidate.load_state_dict(sd, strict=True)
        except RuntimeError as e:
            print(f'  {name} convert={convert}: {str(e)[:260].replace(chr(10), " ")}')
            continue
        model, loaded_as = candidate, f'{name} convert={convert}'
        break
    if model is not None:
        break
assert model is not None, 'the checkpoint does not load strictly into any timm DINOv3 ViT-L variant'
print('LOADED strictly as', loaded_as, '| prefix tokens', model.num_prefix_tokens, '| features', model.num_features)

backbone = PooledBackbone(model).cuda()


In [ ]:
# Throughput, in the three places it can be lost: decoding the stored JPEGs, the model, and the two together.
from torch.utils.data import DataLoader

ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=None, n_groups=9, out_size=336,
                        slots=slot_names, augment=False)
loader = DataLoader(ds, batch_size=None, shuffle=False, num_workers=4, prefetch_factor=2)

t0 = time.time(); n_images = 0; present_counts = []
for image, mask, labels, uid in loader:
    n_images += int(mask.sum()) * image.shape[1]
    present_counts.append(int(mask.sum()))
decode_s = time.time() - t0
print(f'decode only: {decode_s:.1f}s for {len(uids)} studies = {decode_s / len(uids):.3f}s/study, '
      f'{n_images / len(uids):.1f} present images/study')

batch = torch.rand(48, 3, 336, 336, device='cuda')
with torch.autocast('cuda', dtype=torch.float16):
    for _ in range(2):
        backbone(batch)
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(5):
        backbone(batch)
    torch.cuda.synchronize()
img_per_s = 5 * 48 / (time.time() - t0)
print(f'model only: {img_per_s:.1f} images/s at 336px, peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')

feats, masks = [], []
t0 = time.time()
for image, mask, labels, uid in loader:
    with torch.autocast('cuda', dtype=torch.float16):
        f = extract_study_features(backbone, image.cuda(non_blocking=True), mask.cuda())
    feats.append(f.float().cpu().numpy().astype(np.float16)); masks.append(mask.numpy())
torch.cuda.synchronize()
e2e_s = time.time() - t0
per_study = e2e_s / len(uids)
print(f'end to end: {e2e_s:.1f}s = {per_study:.3f}s/study')
print(f'PROJECTED  corpus (4,407 studies): {per_study * 4407 / 3600:.2f} h | test (~1,300 studies): {per_study * 1300 / 60:.1f} min')


In [ ]:
# The cache, and two sanity checks that a frozen backbone is doing something.
feats = np.stack(feats); masks = np.stack(masks)
assert np.isfinite(feats.astype(np.float32)).all(), 'non-finite features in fp16'
os.makedirs('/kaggle/working/features', exist_ok=True)
CACHE = '/kaggle/working/features/pilot_dinov3l_336.npz'
save_feature_cache(CACHE, feats, masks, uids, backbone='dinov3_vit_large_patch16@336')
cache = load_feature_cache(CACHE, expect_uids=uids)
print(f'cache {cache.features.shape} {cache.features.dtype}, {os.path.getsize(CACHE) / 2**20:.0f} MB for {len(uids)} studies '
      f'-> corpus ~{os.path.getsize(CACHE) / 2**20 * 4407 / len(uids) / 1024:.2f} GB')

f = cache.features.astype(np.float32)
def cos(a, b):
    a = a / np.linalg.norm(a, axis=-1, keepdims=True).clip(1e-6); b = b / np.linalg.norm(b, axis=-1, keepdims=True).clip(1e-6)
    return float((a * b).sum(-1).mean())
within, between = [], []
for i in range(len(uids)):
    slots_present = np.where(cache.mask[i])[0]
    if len(slots_present) == 0:
        continue
    s = slots_present[0]
    within.append(cos(f[i, s, 0], f[i, s, 4]))
    j = (i + 1) % len(uids)
    if cache.mask[j, s]:
        between.append(cos(f[i, s, 4], f[j, s, 4]))
print(f'cosine, same slot, different window of ONE study: {np.mean(within):.3f} | same slot and window, DIFFERENT studies: {np.mean(between):.3f}')
print('present slots per study: mean %.2f' % cache.mask.sum(1).mean())
print('PILOT DONE | weights', loaded_as)
